In [55]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import optuna
from catboost import CatBoostClassifier

c:\Users\varva\Desktop\study\credit_score\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Очистка данных

Из результатов 1_initial_eda.ipynb: заполняем наны, удаляем дубликаты, удаляем неинформативные колонки, удаяем выбросы, логарифмируем признаки

In [126]:
data = pd.read_csv('../data/credit_risk_dataset.csv')

print('Размер датасета до изменений:', data.shape)

data = data.drop_duplicates()

data = data.drop(columns=['person_age', 'cb_person_cred_hist_length'])

data['person_income'] = np.log1p(data['person_income'])
data['loan_amnt'] = np.log1p(data['loan_amnt'])
data['person_emp_length'] = np.log1p(data['person_emp_length'])


Размер датасета до изменений: (32581, 12)


In [127]:
cat_cols = data.select_dtypes(include=['str']).columns.to_list()
num_cols = data.select_dtypes(exclude=['str']).columns.to_list()

num_cols.remove('loan_status')

## Разделениие на train/val/test, скейлинг и энкодинг категориальных переменных

для бейзлайна сдклаем OHE, ну а кэтбуст сам разберется со своим внутренним traget encoding)

In [128]:
X = data.drop(columns=['loan_status'])
y = data['loan_status']

X_encoded = pd.get_dummies(X)

column_names = X_encoded.columns


X_train, X_tmp, y_train, y_tmp = train_test_split(X_encoded, y, test_size=0.2, stratify=y, random_state=21)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=column_names, index=X_train.index)
X_val = pd.DataFrame(scaler.transform(X_val), columns=column_names, index=X_val.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=column_names, index=X_test.index)


заполнение пропусков

In [129]:
X_train['person_emp_length'] = X_train['person_emp_length'].fillna(X_train['person_emp_length'].median())
X_train['loan_int_rate'] = X_train['loan_int_rate'].fillna(X_train['loan_int_rate'].median())

X_val['person_emp_length'] = X_val['person_emp_length'].fillna(X_val['person_emp_length'].median())
X_val['loan_int_rate'] = X_val['loan_int_rate'].fillna(X_val['loan_int_rate'].median())

X_test['person_emp_length'] = X_test['person_emp_length'].fillna(X_test['person_emp_length'].median())
X_test['loan_int_rate'] = X_test['loan_int_rate'].fillna(X_test['loan_int_rate'].median())

удаление выбросов

In [130]:
for col in num_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    X_train = X_train[(X_train[col] >= Q1 - 1.5*IQR) & (X_train[col] <= Q3 + 1.5*IQR)]
    
y_train = y_train.loc[X_train.index]

## Бейзлайн

In [131]:
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=21)

logreg.fit(X_train, y_train)

y_pred_val = logreg.predict(X_val)
y_pred_proba_val = logreg.predict_proba(X_val)

print('Precision:', precision_score(y_val, y_pred_val))
print('Recall:', recall_score(y_val, y_pred_val))
print('F1-score:', f1_score(y_val, y_pred_val))
print('ROC-AUC:', roc_auc_score(y_val, y_pred_proba_val[:, 1]))


Precision: 0.5370018975332068
Recall: 0.7983074753173484
F1-score: 0.6420873511060692
ROC-AUC: 0.8794741569254808


roc_auc получился нормальный, а вот precision страдает. Из-за большого количества ложноположительных результатов банк будет упускать много прибыли.

In [54]:
logreg_scores = pd.Series(data=np.abs(logreg.coef_[0]), index=column_names)
print(logreg_scores.sort_values(ascending=False))

loan_percent_income               1.650407
loan_amnt                         1.100904
loan_grade_D                      0.560878
person_home_ownership_OWN         0.542186
loan_int_rate                     0.353778
loan_grade_E                      0.328677
loan_intent_VENTURE               0.304231
person_home_ownership_RENT        0.264827
loan_grade_A                      0.239242
person_income                     0.236374
loan_grade_G                      0.222609
loan_grade_B                      0.217018
loan_intent_HOMEIMPROVEMENT       0.215304
loan_grade_F                      0.182532
loan_intent_MEDICAL               0.126447
loan_intent_DEBTCONSOLIDATION     0.122472
loan_grade_C                      0.111794
loan_intent_EDUCATION             0.108564
person_emp_length                 0.028556
person_home_ownership_MORTGAGE    0.018532
person_home_ownership_OTHER       0.008586
loan_intent_PERSONAL              0.008158
cb_person_default_on_file_Y       0.001426
cb_person_d

для логистической регрессии самыми информативным признаками оказались процент дохода, который составляет сумма кредита и сумма кредита

## Обучение основной модели

для catboost не нужно скейлить и не нужно делать энкодинг

In [132]:
X_cat = data.drop(columns=['loan_status'])
y_cat = data['loan_status']

X_train_cat, X_tmp_cat, y_train_cat, y_tmp_cat = train_test_split(
    X_cat, y_cat, test_size=0.2, stratify=y_cat, random_state=21
)
X_val_cat, X_test_cat, y_val_cat, y_test_cat = train_test_split(
    X_tmp_cat, y_tmp_cat, test_size=0.5, stratify=y_tmp_cat, random_state=21
)

X_train_cat['person_emp_length'] = X_train_cat['person_emp_length'].fillna(X_train_cat['person_emp_length'].median())
X_train_cat['loan_int_rate'] = X_train_cat['loan_int_rate'].fillna(X_train_cat['loan_int_rate'].median())

X_val_cat['person_emp_length'] = X_val_cat['person_emp_length'].fillna(X_val_cat['person_emp_length'].median())
X_val_cat['loan_int_rate'] = X_val_cat['loan_int_rate'].fillna(X_val_cat['loan_int_rate'].median())

X_test_cat['person_emp_length'] = X_test_cat['person_emp_length'].fillna(X_test_cat['person_emp_length'].median())
X_test_cat['loan_int_rate'] = X_test_cat['loan_int_rate'].fillna(X_test_cat['loan_int_rate'].median())

for col in num_cols:
    Q1 = X_train_cat[col].quantile(0.25)
    Q3 = X_train_cat[col].quantile(0.75)
    IQR = Q3 - Q1
    
    X_train_cat = X_train_cat[(X_train_cat[col] >= Q1 - 1.5*IQR) & (X_train_cat[col] <= Q3 + 1.5*IQR)]
    
y_train_cat = y_train_cat.loc[X_train_cat.index]

cat_features = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

In [60]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10),
        'verbose': False,
        'task_type': 'GPU',      
        'devices': '0',
        'random_seed': 21,
        'cat_features': cat_features,
        'thread_count': -1
    }
    
    model = CatBoostClassifier(**params)
    model.fit(X_train_cat, y_train_cat, eval_set=(X_val_cat, y_val_cat), early_stopping_rounds=50, verbose=False)
    
    y_pred_proba = model.predict_proba(X_val_cat)[:, 1]
    roc_auc = roc_auc_score(y_val_cat, y_pred_proba)
    
    return roc_auc

подбор гиперпараметров:

In [63]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Лучшие параметры: {study.best_params}")
print(f"Лучший ROC-AUC: {study.best_value:.4f}")

[I 2026-03-14 22:10:25,103] A new study created in memory with name: no-name-ba50a1e3-14b2-4f51-968d-625b4d64feea
Best trial: 0. Best value: 0.931841:   2%|▏         | 1/50 [00:20<16:33, 20.28s/it]

[I 2026-03-14 22:10:45,374] Trial 0 finished with value: 0.9318410214650117 and parameters: {'iterations': 219, 'depth': 7, 'learning_rate': 0.019179330471750227, 'l2_leaf_reg': 0.008606141053418476, 'random_strength': 2.658138036615367, 'bagging_temperature': 0.049633840438641164, 'border_count': 214, 'scale_pos_weight': 2.081228359776521}. Best is trial 0 with value: 0.9318410214650117.


Best trial: 1. Best value: 0.933117:   4%|▍         | 2/50 [01:32<40:40, 50.83s/it]

[I 2026-03-14 22:11:57,604] Trial 1 finished with value: 0.9331168494573132 and parameters: {'iterations': 943, 'depth': 9, 'learning_rate': 0.014281841341478585, 'l2_leaf_reg': 0.4460638120801849, 'random_strength': 0.0038956579591936475, 'bagging_temperature': 0.06484551178041642, 'border_count': 253, 'scale_pos_weight': 6.9202060224843205}. Best is trial 1 with value: 0.9331168494573132.


Best trial: 2. Best value: 0.938306:   6%|▌         | 3/50 [01:44<25:55, 33.09s/it]

...

[I 2026-03-14 22:36:55,728] Trial 47 finished with value: 0.9519975988348663 and parameters: {'iterations': 444, 'depth': 5, 'learning_rate': 0.10008241922040348, 'l2_leaf_reg': 2.289009190622256, 'random_strength': 0.0011375627425264268, 'bagging_temperature': 0.27553038339851543, 'border_count': 220, 'scale_pos_weight': 6.193460561106554}. Best is trial 47 with value: 0.9519975988348663.


Best trial: 47. Best value: 0.951998:  98%|█████████▊| 49/50 [26:44<00:13, 13.26s/it]

[I 2026-03-14 22:37:09,431] Trial 48 finished with value: 0.9501532498499271 and parameters: {'iterations': 428, 'depth': 5, 'learning_rate': 0.09972183298510966, 'l2_leaf_reg': 0.5541601455969929, 'random_strength': 0.0034641977497128535, 'bagging_temperature': 0.27665518918702975, 'border_count': 219, 'scale_pos_weight': 6.066541136319734}. Best is trial 47 with value: 0.9519975988348663.


Best trial: 49. Best value: 0.953569: 100%|██████████| 50/50 [27:00<00:00, 32.41s/it]

[I 2026-03-14 22:37:25,349] Trial 49 finished with value: 0.9535693921674527 and parameters: {'iterations': 431, 'depth': 5, 'learning_rate': 0.10763965306425233, 'l2_leaf_reg': 2.116248801552587, 'random_strength': 0.0011349641784642202, 'bagging_temperature': 0.310611777141209, 'border_count': 221, 'scale_pos_weight': 8.24709157220298}. Best is trial 49 with value: 0.9535693921674527.
Лучшие параметры: {'iterations': 431, 'depth': 5, 'learning_rate': 0.10763965306425233, 'l2_leaf_reg': 2.116248801552587, 'random_strength': 0.0011349641784642202, 'bagging_temperature': 0.310611777141209, 'border_count': 221, 'scale_pos_weight': 8.24709157220298}
Лучший ROC-AUC: 0.9536


In [133]:
best_params = study.best_params
best_params.update({
    'verbose': 100,
    'random_seed': 21,
    'cat_features': cat_features,
    'thread_count': -1,
    'task_type': 'GPU',      
    'devices': '0',
})

catboost_final = CatBoostClassifier(**best_params)
catboost_final.fit(X_train_cat, y_train_cat, eval_set=(X_val_cat, y_val_cat), early_stopping_rounds=50)

y_pred_val_cat = catboost_final.predict(X_val_cat)
y_pred_proba_val_cat = catboost_final.predict_proba(X_val_cat)

print("=== CatBoost на валидации ===")
print(f"Precision: {precision_score(y_val_cat, y_pred_val_cat):.4f}")
print(f"Recall: {recall_score(y_val_cat, y_pred_val_cat):.4f}")
print(f"F1-score: {f1_score(y_val_cat, y_pred_val_cat):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val_cat, y_pred_proba_val_cat[:, 1]):.4f}")

0:	learn: 0.6274929	test: 0.6233364	best: 0.6233364 (0)	total: 25.6ms	remaining: 11s
100:	learn: 0.2818771	test: 0.2886978	best: 0.2878326 (90)	total: 3.1s	remaining: 10.1s
200:	learn: 0.2498500	test: 0.2751484	best: 0.2744292 (193)	total: 6.23s	remaining: 7.13s
300:	learn: 0.2279343	test: 0.2713050	best: 0.2700429 (281)	total: 9.29s	remaining: 4.01s
bestTest = 0.2700429142
bestIteration = 281
Shrink model to first 282 iterations.
=== CatBoost на валидации ===
Precision: 0.5964
Recall: 0.8773
F1-score: 0.7100
ROC-AUC: 0.9457


In [90]:
feature_importance = pd.Series(
    catboost_final.get_feature_importance(),
    index=X_train_cat.columns
).sort_values(ascending=False)

print("Топ-10 важных признаков:")
print(feature_importance.head(10))

Топ-10 важных признаков:
loan_percent_income          40.188907
person_income                19.752919
person_home_ownership        13.512387
loan_intent                   8.905198
person_emp_length             6.361359
loan_grade                    4.342905
loan_amnt                     4.113812
loan_int_rate                 2.716439
cb_person_default_on_file     0.106075
dtype: float64


## Подбор порога (еще нужно доработать)

In [134]:
y_proba = catboost_final.predict_proba(X_val_cat)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = []

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_val_cat, y_pred))

best_threshold_f1 = thresholds[np.argmax(f1_scores)]

y_pred_optimized = (y_proba >= best_threshold_f1).astype(int)
print(f"Лучший порог по F1: {best_threshold_f1:.3f}")
print(f"F1 при этом: {max(f1_scores):.4f}")
print(f"Precision при этом: {precision_score(y_val_cat, y_pred_optimized)}")
print(f"Recall при этом: {recall_score(y_val_cat, y_pred_optimized)}")

Лучший порог по F1: 0.743
F1 при этом: 0.8261
Precision при этом: 0.9043624161073825
Recall при этом: 0.7602256699576869


## Итого:

In [135]:
y_proba = catboost_final.predict_proba(X_train_cat)[:, 1]
y_pred_train_final = (y_proba >= best_threshold_f1).astype(int)
print("=== CatBoost на трейне ===")
print(f"Precision: {precision_score(y_train_cat, y_pred_train_final):.4f}")
print(f"Recall: {recall_score(y_train_cat,  y_pred_train_final):.4f}")
print(f"F1-score: {f1_score(y_train_cat,  y_pred_train_final):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_train_cat, y_proba):.4f}")

y_proba = catboost_final.predict_proba(X_val_cat)[:, 1]
y_pred_val_final = (y_proba >= best_threshold_f1).astype(int)
print("\n\n=== CatBoost на валидации ===")
print(f"Precision: {precision_score(y_val_cat, y_pred_val_final):.4f}")
print(f"Recall: {recall_score(y_val_cat,  y_pred_val_final):.4f}")
print(f"F1-score: {f1_score(y_val_cat,  y_pred_val_final):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val_cat, y_proba):.4f}")

y_proba = catboost_final.predict_proba(X_test_cat)[:, 1]
y_pred_test_final = (y_proba >= best_threshold_f1).astype(int)
print("\n\n=== CatBoost на тесте ===")
print(f"Precision: {precision_score(y_test_cat, y_pred_test_final):.4f}")
print(f"Recall: {recall_score(y_test_cat,  y_pred_test_final):.4f}")
print(f"F1-score: {f1_score(y_test_cat,  y_pred_test_final):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_cat, y_proba):.4f}")

=== CatBoost на трейне ===
Precision: 0.8939
Recall: 0.7746
F1-score: 0.8300
ROC-AUC: 0.9677


=== CatBoost на валидации ===
Precision: 0.9044
Recall: 0.7602
F1-score: 0.8261
ROC-AUC: 0.9457


=== CatBoost на тесте ===
Precision: 0.8830
Recall: 0.7560
F1-score: 0.8146
ROC-AUC: 0.9460


### Что нужно доработать:
roc-auc 0.94 конечно идеализированное значение, но на то у нас и учебные данные  
необходимо доработать калибровку вероятностей, чтобы сотрудники банка могли видеть более честную картину  
необходимо сделать правильный подбор порога под риск-аппетит банка

в принципе 'loan_int_rate' и 'loan_grade' не особо можно использовать, так как это утечка и процентная ставка назначается банком скоринга, а рейтинг присваивается банком на основе внутренней оценки риска, но в датасете они почему-то есть    


In [138]:
data['person_home_ownership'].unique()

<StringArray>
['RENT', 'OWN', 'MORTGAGE', 'OTHER']
Length: 4, dtype: str